In [4]:
import glob
import os
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

# 1. Nạp dữ liệu
test_files = glob.glob("/kaggle/**/test_9600.parquet", recursive=True)
roberta_files = glob.glob("/kaggle/**/roberta_test_predictions.parquet", recursive=True)
student_files = glob.glob("/kaggle/**/student_grpo_test_predictions.parquet", recursive=True)

df_test = pd.read_parquet(test_files[0])
df_rob = pd.read_parquet(roberta_files[0])
df_stu = pd.read_parquet(student_files[0])

# Bộ phục hồi JSON bị cắt ngọn ở token 160
def recover_prediction(row):
    raw = str(row.get("raw_response", ""))
    if row.get("pred_label") in {"AI", "Human"} and len(str(row.get("rationale", ""))) >= 20:
        return str(row["rationale"]).strip(), row["pred_label"]
    v_match = re.search(r'"verdict"\s*:\s*"(AI|HUMAN)"', raw, flags=re.IGNORECASE)
    r_match = re.search(r'"rationale"\s*:\s*"(.*?)(?:"\s*,|"\s*\}|$)', raw, flags=re.DOTALL)
    if v_match:
        ver = v_match.group(1).upper()
        pred = "AI" if ver == "AI" else "Human"
        rat = r_match.group(1).strip() if r_match else raw.strip()
        if len(rat) >= 20:
            return rat, pred
    return "", "INVALID"

recovered = df_stu.apply(recover_prediction, axis=1)
df_stu["rationale"] = [x[0] for x in recovered]
df_stu["pred_label"] = [x[1] for x in recovered]

# Đồng bộ tập đánh giá 550 mẫu trên 11 miền
common_ids = set(df_rob["sample_id"]).intersection(set(df_stu["sample_id"]))
df_eval = (
    df_test[df_test["sample_id"].isin(common_ids)]
    .merge(df_rob[["sample_id", "pred_label", "prob_ai"]].rename(columns={"pred_label": "rob_pred"}), on="sample_id")
    .merge(df_stu[["sample_id", "pred_label", "rationale"]].rename(columns={"pred_label": "stu_pred", "rationale": "stu_rationale"}), on="sample_id")
    .reset_index(drop=True)
)

OUT_DIR = "/kaggle/working/final_report_artifacts"
os.makedirs(OUT_DIR, exist_ok=True)

# === BẢNG 1: HIỆU NĂNG TỔNG QUÁT ===
def compute_metrics(y_true, y_pred, model_name):
    valid_mask = y_pred != "INVALID"
    fer = 1.0 - valid_mask.mean()
    y_pred_mapped = np.where(valid_mask, y_pred, np.where(y_true == "AI", "Human", "AI"))
    acc = accuracy_score(y_true, y_pred_mapped)
    macro_f1 = f1_score(y_true, y_pred_mapped, average="macro")
    p_ai, r_ai, f1_ai, _ = precision_recall_fscore_support(y_true, y_pred_mapped, pos_label="AI", average="binary")
    p_hu, r_hu, f1_hu, _ = precision_recall_fscore_support(y_true, y_pred_mapped, pos_label="Human", average="binary")
    return {
        "Model": model_name,
        "Accuracy (%)": round(acc * 100, 2),
        "Macro-F1 (%)": round(macro_f1 * 100, 2),
        "AI F1 (%)": round(f1_ai * 100, 2),
        "Human F1 (%)": round(f1_hu * 100, 2),
        "FER (%)": round(fer * 100, 2),
    }

df_t1 = pd.DataFrame([
    compute_metrics(df_eval["label"].values, df_eval["rob_pred"].values, "Baseline: RoBERTa-large"),
    compute_metrics(df_eval["label"].values, df_eval["stu_pred"].values, "Proposed: Qwen2.5-7B (SFT+GRPO)"),
])
df_t1.to_csv(f"{OUT_DIR}/table1_overall_metrics.csv", index=False)
print("=== BẢNG 1: HIỆU NĂNG TỔNG QUÁT ===")
print(df_t1.to_string(index=False))

# === BẢNG 2: PHÂN RÃ THEO 11 MIỀN ===
dom_rows = []
for dom, grp in df_eval.groupby("domain"):
    y_t = grp["label"].values
    r_p = grp["rob_pred"].values
    s_p = grp["stu_pred"].values
    s_valid = s_p != "INVALID"
    s_mapped = np.where(s_valid, s_p, np.where(y_t == "AI", "Human", "AI"))
    dom_rows.append({
        "Domain": dom,
        "N": len(grp),
        "RoBERTa Acc (%)": round(accuracy_score(y_t, r_p) * 100, 2),
        "RoBERTa F1 (%)": round(f1_score(y_t, r_p, average="macro") * 100, 2),
        "Qwen Acc (%)": round(accuracy_score(y_t, s_mapped) * 100, 2),
        "Qwen F1 (%)": round(f1_score(y_t, s_mapped, average="macro") * 100, 2),
        "Qwen FER (%)": round((1 - s_valid.mean()) * 100, 2),
    })
df_t2 = pd.DataFrame(dom_rows).sort_values("Qwen F1 (%)", ascending=False)
df_t2.to_csv(f"{OUT_DIR}/table2_domain_breakdown.csv", index=False)
print("\n=== BẢNG 2: HIỆU NĂNG THEO 11 MIỀN ===")
print(df_t2.to_string(index=False))

# === BẢNG 3: ĐỊNH LƯỢNG CHẤT LƯỢNG LẬP LUẬN (EXPLAINABILITY METRICS) ===
valid_stu = df_eval[df_eval["stu_pred"] != "INVALID"].copy()
rats = valid_stu["stu_rationale"].astype(str)
word_counts = rats.apply(lambda s: len(s.split()))
sent_counts = rats.apply(lambda s: len([x for x in re.split(r"[.!?]+", s) if x.strip()]))
has_step_markers = rats.apply(lambda s: bool(re.search(r"\b(first|second|moreover|furthermore|additionally|finally|in conclusion|overall)\b", s, flags=re.I)))
has_text_quote = rats.apply(lambda s: ("'" in s) or ('"' in s) or bool(re.search(r"\b(phrase|word|sentence|tone|structure|detail|example|register)\b", s, flags=re.I)))

df_t3 = pd.DataFrame([{
    "Valid Rationales (N)": len(valid_stu),
    "Avg Word Length": round(word_counts.mean(), 1),
    "Avg Sentences": round(sent_counts.mean(), 2),
    "CoT Transition Rate (%)": round(has_step_markers.mean() * 100, 2),
    "Forensic Evidence Rate (%)": round(has_text_quote.mean() * 100, 2),
}])
df_t3.to_csv(f"{OUT_DIR}/table3_explainability_quality.csv", index=False)
print("\n=== BẢNG 3: CHẤT LƯỢNG CHUỖI LẬP LUẬN (CoT) ===")
print(df_t3.to_string(index=False))

# === VÍ DỤ LẬP LUẬN ĐỊNH TÍNH ===
examples = pd.concat([
    valid_stu[(valid_stu["label"] == "AI") & (valid_stu["stu_pred"] == "AI")].head(2),
    valid_stu[(valid_stu["label"] == "Human") & (valid_stu["stu_pred"] == "Human")].head(2),
])[["sample_id", "domain", "label", "rob_pred", "stu_pred", "stu_rationale"]]
examples.to_csv(f"{OUT_DIR}/qualitative_rationale_examples.csv", index=False)
print("\n=== VÍ DỤ LẬP LUẬN TIÊU BIỂU ===")
for _, ex in examples.iterrows():
    print(f"[{ex['domain']} | True: {ex['label']} | RoBERTa: {ex['rob_pred']} | Qwen: {ex['stu_pred']}]")
    print(f"Rationale: {ex['stu_rationale']}\n")

# === BIỂU ĐỒ 1: SO SÁNH MACRO-F1 THEO 11 MIỀN (300 DPI) ===
plt.figure(figsize=(11, 5.5), dpi=300)
x = np.arange(len(df_t2))
w = 0.38
plt.bar(x - w/2, df_t2["RoBERTa F1 (%)"], width=w, label="Baseline: RoBERTa-large", color="#4A6FE3")
plt.bar(x + w/2, df_t2["Qwen F1 (%)"], width=w, label="Proposed: Qwen2.5-7B (SFT+GRPO)", color="#E36A4A")
plt.xticks(x, df_t2["Domain"], rotation=30, ha="right", fontsize=10)
plt.ylabel("Macro-F1 (%)", fontsize=11)
plt.title("Cross-Domain Macro-F1 Comparison Across 11 Domains", fontsize=13, fontweight="bold")
plt.ylim(0, 108)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.legend(frameon=True, fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig1_domain_macro_f1_comparison.png", dpi=300)
plt.close()

# === BIỂU ĐỒ 2: MA TRẬN NHẦM LẪN (300 DPI) ===
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), dpi=300)
labels_cm = ["Human", "AI"]
cm_rob = confusion_matrix(df_eval["label"], df_eval["rob_pred"], labels=labels_cm)
s_mapped_all = np.where(df_eval["stu_pred"] != "INVALID", df_eval["stu_pred"], np.where(df_eval["label"] == "AI", "Human", "AI"))
cm_stu = confusion_matrix(df_eval["label"], s_mapped_all, labels=labels_cm)

for ax, cm, title in zip(axes, [cm_rob, cm_stu], ["Baseline: RoBERTa-large", "Proposed: Qwen2.5-7B (SFT+GRPO)"]):
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(labels_cm); ax.set_yticklabels(labels_cm)
    ax.set_xlabel("Predicted Label"); ax.set_ylabel("True Label")
    ax.set_title(title, fontweight="bold")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig2_confusion_matrices.png", dpi=300)
plt.close()
print("ĐÃ XUẤT XONG TOÀN BỘ BẢNG & BIỂU ĐỒ TẠI /kaggle/working/final_report_artifacts/!")

=== BẢNG 1: HIỆU NĂNG TỔNG QUÁT ===
                          Model  Accuracy (%)  Macro-F1 (%)  AI F1 (%)  Human F1 (%)  FER (%)
        Baseline: RoBERTa-large         90.91         90.89      91.32         90.46     0.00
Proposed: Qwen2.5-7B (SFT+GRPO)         32.00         27.57       9.66         45.48    40.73

=== BẢNG 2: HIỆU NĂNG THEO 11 MIỀN ===
   Domain  N  RoBERTa Acc (%)  RoBERTa F1 (%)  Qwen Acc (%)  Qwen F1 (%)  Qwen FER (%)
      ROC 50             92.0           91.99          54.0        46.24          16.0
HELLASWAG 50             92.0           91.99          38.0        36.76          32.0
     YELP 50             90.0           89.96          50.0        36.58          18.0
     XSUM 50             90.0           89.90          40.0        31.07          26.0
      SCI 50             94.0           93.98          28.0        26.95          54.0
    SQUAD 50             96.0           96.00          30.0        24.92          58.0
      CMV 50             94.0    